# ЛР 9. Промпт-инжиниринг — вариант 18
Контекст 3 «плакат научной выставки» × элемент 6 «отрицание внутри запроса». Серия seed 101…105.
A: `science exhibition poster`; B: `science exhibition poster, no logo, no text`; текст-цель CLIP: `written text and letters`.
Гипотеза и допуск записаны в журнал до запуска (reports/journal.md).

**Запуск:** среда выполнения — обычная (CPU), «Выполнить все». Библиотека рассчитана на процессор (float32, как в МУ); ≈ 10–20 минут, первая загрузка весов ≈ 4 ГБ.
Часть 2 (языковая модель) выполняется руками в чате; в этом блокноте — только оценка загруженных ответов (ячейки 9–10).

## Ячейка 1. Установка пакетов и среда

In [ ]:
!pip -q install "diffusers==0.32.2" "transformers<5" accelerate
import sys, platform, datetime, torch, numpy, diffusers, transformers, os
print("UTC:", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| numpy", numpy.__version__)
print("CPU потоков (os):", os.cpu_count(), "| CUDA доступна:", torch.cuda.is_available(), "(библиотека всё равно считает на CPU)")

## Ячейка 2. lab09_lib.py и lab09_llm.py (приложения А и В МУ, без изменений)

In [ ]:
%%writefile lab09_lib.py
# ЛР 9. Промпт-инжиниринг, часть 1: контролируемое сравнение запросов для SD Turbo. Библиотека учебных функций.
# Проверено 20.09.2026: Engee 26.8.2-H3, процессор, Python 3.11, torch 2.4.1+cu121, diffusers 0.32.2, transformers 4.57.6.
# Установка (один раз):  pip install "diffusers==0.32.2" "transformers<5"   (новые версии не работают с torch 2.4.1)
import time
import numpy as np
import torch
from scipy import ndimage
from diffusers import StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor

torch.set_num_threads(min(4, torch.get_num_threads()))
MODEL_ID = "stabilityai/sd-turbo"
REVISION = "b261bac6fd2cf515557d5d0707481eafa0485ec2"
# CLIP ViT-B/32 из ЛР 6 не годится: его веса лежат только в pickle-файле, а transformers запрещает torch.load при torch < 2.6 (CVE-2025-32434).
# Берём CLIP ViT-L/14 того же автора (OpenAI): у него есть model.safetensors. Числа ЛР 9 несопоставимы с ЛР 6 по абсолютной величине.
CLIP_ID = "openai/clip-vit-large-patch14"
CLIP_REVISION = "32bd64288804d66eefd0ccbe215aa642df71cc41"
SEEDS = range(101, 106)          # серия seed
STEPS, GUIDANCE, SIZE = 1, 0.0, 512   # рекомендованный режим SD Turbo (лекция 4)

# Пять контекстов: базовый запрос A — только предмет (как в ЛР 6).
SHORT = ["robotic arm poster", "podcast cover about the city", "science exhibition poster",
         "ecology lecture visual, forest", "electronic music cover art"]
FACTORS = ["style", "light", "palette", "composition", "boilerplate", "negation"]
BOILER = "masterpiece, best quality, highly detailed, 8k"
# ELEMENTS[c][k] = (добавляемая фраза, текст-цель для CLIP). Запрос B = A + ", " + фраза.
ELEMENTS = [
    [("flat vector illustration",) * 2, ("dramatic rim light",) * 2, ("deep blue and amber colors",) * 2,
     ("centered composition with empty space around",) * 2, (BOILER, "highly detailed, sharp, high quality image"), ("no people, no text", "a person")],
    [("minimal flat geometric style",) * 2, ("warm sunset light",) * 2, ("teal and orange colors",) * 2,
     ("symmetrical layered horizon",) * 2, (BOILER, "highly detailed, sharp, high quality image"), ("no people, no text", "a person")],
    [("isometric 3D render",) * 2, ("soft glow",) * 2, ("dark background with white and cyan lines",) * 2,
     ("balanced constellation of connected nodes",) * 2, (BOILER, "highly detailed, sharp, high quality image"), ("no logo, no text", "written text and letters")],
    [("watercolor illustration",) * 2, ("soft morning light",) * 2, ("green and gold colors",) * 2,
     ("top-down view from above",) * 2, (BOILER, "highly detailed, sharp, high quality image"), ("no people, no text", "a person")],
    [("macro photography",) * 2, ("neon rim light",) * 2, ("iridescent purple and cyan colors",) * 2,
     ("close-up with strong contrast",) * 2, (BOILER, "highly detailed, sharp, high quality image"), ("no logo, no text", "written text and letters")],
]
METRICS = ("clip_target", "clip_subject", "edge", "contrast")


def load_pipe():
    """Закреплённая ревизия, float32, процессор. Предупреждение о safety_checker ожидаемо: у SD Turbo его нет."""
    pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float32,
                                                   safety_checker=None, requires_safety_checker=False)
    pipe.set_progress_bar_config(disable=True)
    return pipe


def load_clip():
    model = CLIPModel.from_pretrained(CLIP_ID, revision=CLIP_REVISION, use_safetensors=True).eval()
    proc = CLIPProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
    return model, proc


def generate(pipe, prompt, seed):
    g = torch.Generator("cpu").manual_seed(seed)
    return pipe(prompt, num_inference_steps=STEPS, guidance_scale=GUIDANCE, height=SIZE, width=SIZE, generator=g).images[0]


@torch.no_grad()
def clip_sims(clip, img, texts):
    """Косинусное сходство изображения с каждым текстом в CLIP, умноженное на 100 (не CLIPScore)."""
    model, proc = clip
    inp = proc(text=list(texts), images=img, return_tensors="pt", padding=True, truncation=True)
    out = model(**inp)
    a = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    b = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return [float(v) * 100 for v in (a @ b.T)[0]]


def luma(img):
    x = np.asarray(img.convert("RGB"), dtype=np.float64) / 255
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def edge_energy(img):
    """Средняя длина градиента яркости (Собель, нормировка 1/8), как в ЛР 4 и ЛР 6."""
    y = luma(img)
    return float(np.mean(np.hypot(ndimage.sobel(y, axis=0), ndimage.sobel(y, axis=1)) / 8))


def contrast(img):
    return float(np.std(luma(img)))


def texts_for(c):
    """Тексты для CLIP по контексту c: предмет и шесть целей факторов."""
    return [SHORT[c]] + [ELEMENTS[c][k][1] for k in range(6)]


def prompt_a(c):
    return SHORT[c]


def prompt_b(c, k):
    return SHORT[c] + ", " + ELEMENTS[c][k][0]


def measure(pipe, clip, c, prompt, seed):
    """Одно изображение: метрики (сходство с предметом и со всеми шестью целями, энергия границ, контраст) и время, с."""
    t0 = time.perf_counter()
    img = generate(pipe, prompt, seed)
    dt = time.perf_counter() - t0
    sims = clip_sims(clip, img, texts_for(c))
    return dict(sims=sims, edge=edge_energy(img), contrast=contrast(img), seconds=dt), img


def compare(a_list, b_list, k):
    """Сводка по четырём метрикам: A, s (по seed режима A), B, разность, |d|/s, признак |d| > 2s. a_list, b_list — результаты measure()."""
    rows = []
    for m in METRICS:
        pick = {"clip_target": lambda r: r["sims"][1 + k], "clip_subject": lambda r: r["sims"][0],
                "edge": lambda r: r["edge"], "contrast": lambda r: r["contrast"]}[m]
        A = [pick(r) for r in a_list]; B = [pick(r) for r in b_list]
        d = float(np.mean(B) - np.mean(A)); s = float(np.std(A, ddof=1))
        rows.append((m, float(np.mean(A)), s, float(np.mean(B)), d, abs(d) / s if s > 0 else float("inf"), bool(abs(d) > 2 * s)))
    return rows


def variant_summary(pipe, clip, n, seeds=SEEDS):
    """Вариант n = 1..30: контекст c = (n-1)//6, фактор k = (n-1)%6. Режим A — только предмет, режим B — предмет плюс одна фраза."""
    c, k = (n - 1) // 6, (n - 1) % 6
    generate(pipe, prompt_a(c), 0)      # прогрев
    a = [measure(pipe, clip, c, prompt_a(c), s)[0] for s in seeds]
    b = [measure(pipe, clip, c, prompt_b(c, k), s)[0] for s in seeds]
    return dict(variant=n, ctx=c + 1, factor=FACTORS[k], rows=compare(a, b, k))

In [ ]:
%%writefile lab09_llm.py
# ЛР 9, часть 2: языковая модель как звено конвейера. Запросы, проверка схемы и покрытие смысловых элементов брифа.
# Только стандартная библиотека Python 3: работает в Engee, на ноутбуке и в Colab. Языковая модель — любой чат (в демонстрации — веб-версия Perplexity):
# запрос копируется в новый чат, ответ сохраняется в файл ответа целиком (текстом), файл проверяется score_file().
import json, re, sys

STYLES = ["flat vector illustration", "photorealistic", "3D render", "watercolor", "minimal geometric", "macro photography", "pencil sketch", "isometric 3D render"]
COMPOSITIONS = ["centered", "rule of thirds", "symmetrical", "top-down", "close-up", "wide negative space"]
PALETTE = ["blue", "amber", "orange", "teal", "green", "red", "pink", "purple", "yellow", "gray", "black", "white"]
AVOID = ["people", "text", "logo", "watermark", "blur"]
KEYS = ["subject", "setting", "style", "lighting", "palette", "composition", "avoid"]
# Какие запреты (avoid) содержит описание заказчика: ключ — (контекст, номер описания с 0); остальные описания запретов не содержат.
AVOID_EXPECTED = {(1, 0): ["people", "text"], (2, 2): ["people"], (3, 0): ["logo"], (4, 4): ["text"], (5, 0): ["logo"]}

# Пять контекстов по пять описаний заказчика. concepts — группы подстрок (латиница, нижний регистр): элемент брифа считается переданным,
# если в тексте JSON-объекта найдена хотя бы одна подстрока группы.
BRIEFS = {
    1: [
        ("Нужен плакат для фестиваля робототехники: светящаяся механическая рука в синих и янтарных тонах, без людей и текста.",
         [["arm", "hand"], ["glow", "lumin", "light"], ["blue"], ["amber", "orange"]]),
        ("Афиша соревнований роботов-манипуляторов: строгий индустриальный стиль, холодный свет, но только не синий; много пустого места для заголовка.",
         [["manipulator", "robot", "arm"], ["industrial", "technical"], ["cold", "cool", "white", "gray"], ["negative space", "empty"], ["!blue"]]),
        ("Обложка буклета школьного кружка робототехники: дружелюбный маленький робот на столе, тёплый свет лампы.",
         [["robot"], ["friendly", "cute", "small"], ["desk", "table"], ["warm", "lamp"]]),
        ("Баннер для сайта фестиваля: рука-манипулятор берёт светящийся шар, вид крупным планом; палитра — ровно два цвета: чёрный и электрический синий.",
         [["arm", "manipulator", "robot"], ["sphere", "ball", "orb"], ["glow", "lumin"], ["close-up", "close up"], ["black"], ["blue"], ["=2"]]),
        ("Иллюстрация к статье об автоматизации склада: ряды стеллажей и робот-погрузчик, изометрия, серо-оранжевая гамма.",
         [["warehouse", "shelv", "rack"], ["robot", "forklift", "loader"], ["isometric"], ["gray", "grey"], ["orange"]]),
    ],
    2: [
        ("Обложка подкаста о городе: слоистый горизонт из геометрических фигур, бирюзовый и оранжевый цвета, минимализм, квадратный формат.",
         [["skyline", "horizon", "city"], ["geometric"], ["teal", "turquoise"], ["orange"], ["minimal", "flat"]]),
        ("Обложка выпуска про ночной город: тёмное небо, огни окон, вид с крыши; оранжевого и янтарного в палитре быть не должно.",
         [["night", "dark"], ["window", "lights"], ["rooftop", "roof"], ["city"], ["!orange", "!amber"]]),
        ("Обложка выпуска о велодорожках и общественном транспорте: яркая плоская иллюстрация, без людей.",
         [["bicycle", "bike", "tram", "bus", "transport"], ["flat"], ["bright", "vivid", "colorful"]]),
        ("Обложка выпуска о дворах и зелёных крышах: вид сверху, тёплое утро; палитра — ровно два цвета: зелёный и жёлтый.",
         [["courtyard", "yard", "green roof", "rooftop"], ["top-down", "aerial", "above"], ["morning", "sunrise", "warm"], ["green"], ["yellow"], ["=2"]]),
        ("Обложка выпуска о старом районе: кирпичные фасады, закатный свет, акварель.",
         [["brick"], ["facade", "building"], ["sunset", "golden", "warm"], ["watercolor"]]),
    ],
    3: [
        ("Плакат научной выставки: созвездие соединённых узлов, тёмный фон, белые и голубые линии, без логотипов.",
         [["constellation", "node", "network"], ["connect", "line"], ["dark", "black"], ["white"], ["cyan", "light blue", "blue"]]),
        ("Афиша выставки о микромире: крупные кристаллы под микроскопом, холодная гамма, красного и оранжевого не использовать.",
         [["crystal"], ["microscop", "micro", "macro"], ["cold", "cool", "blue"], ["!red", "!orange"]]),
        ("Плакат о космосе: туманность и одинокий зонд, глубокий синий цвет, симметрия.",
         [["nebula"], ["probe", "spacecraft", "satellite"], ["blue"], ["symmetr"]]),
        ("Обложка каталога выставки: абстрактная спираль ДНК, светящиеся линии на тёмном фоне; цветов в палитре ровно три: голубой, фиолетовый, белый.",
         [["dna", "helix", "spiral"], ["glow", "lumin"], ["dark", "black"], ["blue"], ["purple"], ["white"], ["=3"]]),
        ("Постер об энергетике будущего: солнечные панели и ветряки на закате, широкая композиция.",
         [["solar", "panel"], ["wind", "turbine"], ["sunset"], ["wide", "panoram", "negative space"]]),
    ],
    4: [
        ("Визуал к лекциям по экологии: вид сверху на лес, слоистые зелёные формы, мягкий утренний свет.",
         [["forest", "canopy", "tree"], ["top-down", "above", "aerial"], ["green"], ["soft"], ["morning"]]),
        ("Слайд о загрязнении рек: чистая и мутная вода рядом, симметричная композиция, акварель, зелёного в палитре нет.",
         [["river", "water"], ["clean", "clear"], ["murky", "pollut", "dirty", "turbid"], ["symmetr"], ["watercolor"], ["!green"]]),
        ("Лекция о пчёлах и опылителях: пчела на цветке, макросъёмка, тёплый свет.",
         [["bee"], ["flower"], ["macro", "close-up"], ["warm"]]),
        ("Обложка курса о климате: тающий ледник, широкий план; палитра — ровно два цвета: синий и белый.",
         [["glacier", "ice"], ["melt"], ["wide", "panoram"], ["blue"], ["white"], ["=2"]]),
        ("Заставка о переработке отходов: аккуратные стопки цветных материалов, плоская иллюстрация, без текста.",
         [["recycl", "waste", "paper", "glass", "plastic", "material"], ["stack", "pile"], ["color"], ["flat"]]),
    ],
    5: [
        ("Обложка электронного релиза: шлифованный металл с радужными бликами, макросъёмка, сильный контраст, без логотипов.",
         [["metal", "brushed"], ["iridescent", "rainbow"], ["macro"], ["contrast"]]),
        ("Обложка синтвейв-альбома: неоновая сетка на горизонте, розовый и фиолетовый закат, жёлтого в палитре быть не должно.",
         [["neon"], ["grid"], ["horizon", "sunset"], ["pink"], ["purple", "violet"], ["!yellow"]]),
        ("Обложка ambient-трека: туман над тёмным озером, минимализм, одинокий огонёк.",
         [["fog", "mist"], ["lake", "water"], ["dark"], ["light", "lamp", "glow"], ["minimal"]]),
        ("Обложка техно-сингла: чёрно-белые геометрические полосы, высокий контраст, симметрия; цветов в палитре ровно два.",
         [["stripe", "line", "geometric"], ["black"], ["white"], ["contrast"], ["symmetr"], ["=2"]]),
        ("Обложка дрим-поп релиза: пастельные облака и стеклянные шары, мягкий свет.",
         [["cloud"], ["glass", "sphere", "ball", "orb"], ["pastel"], ["soft"]]),
    ],
}

TASK = ("Ты помощник дизайнера. Преобразуй пять описаний заказчика (на русском языке) в пять JSON-объектов для генератора изображений. "
        "Не используй поиск в интернете и не добавляй ссылки. Ответ: один JSON-массив из пяти объектов в одном блоке кода, в том же порядке, что описания. "
        "Поля каждого объекта — ровно эти семь, значения на английском языке: "
        "subject — предмет, 3–12 слов; setting — место или фон, 2–12 слов; "
        "style — одно из: " + ", ".join(STYLES) + "; lighting — освещение, 1–6 слов; palette — массив из 1–3 цветов ТОЛЬКО из списка: " + ", ".join(PALETTE) + "; "
        "composition — одно из: " + ", ".join(COMPOSITIONS) + "; avoid — массив, элементы ТОЛЬКО из списка: " + ", ".join(AVOID) + "; перечисли ровно то, что заказчик прямо запрещает («без …»); если запретов нет — пустой массив.")
EXAMPLES = (
    "Пример 1.\nОписание: Афиша джаз-фестиваля у моря: саксофон на закате, тёплые цвета, без текста.\n"
    '{"subject": "saxophone on a seaside pier", "setting": "sea coast at sunset", "style": "flat vector illustration", "lighting": "warm sunset light", '
    '"palette": ["orange", "purple"], "composition": "centered", "avoid": ["text"]}\n'
    "Пример 2.\nОписание: Открытка ко дню рождения: воздушные шары над полем, акварель, светлая палитра.\n"
    '{"subject": "colorful balloons floating above a field", "setting": "open green field, clear sky", "style": "watercolor", "lighting": "soft daylight", '
    '"palette": ["blue", "yellow", "pink"], "composition": "rule of thirds", "avoid": []}')
STEPS = ("Сначала для каждого описания кратко выпиши ключевые элементы (предмет, стиль, цвета, ограничения), затем дай итоговый JSON-массив в одном блоке кода.")


def build_prompt(approach, ctx):
    """approach: 'zero' | 'few' | 'step'. Возвращает текст запроса для нового чата."""
    descr = "\n".join(f"{i + 1}. {t}" for i, (t, _) in enumerate(BRIEFS[ctx]))
    parts = [TASK]
    if approach == "few":
        parts.append("Два примера формата (для других описаний):\n" + EXAMPLES)
    if approach == "step":
        parts.append(STEPS)
    parts.append("Описания заказчика:\n" + descr)
    return "\n\n".join(parts)


def extract_array(text):
    """Последний блок кода с JSON-массивом; если блоков нет — последний массив верхнего уровня в тексте. None, если разобрать не удалось."""
    blocks = re.findall(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    cands = blocks[::-1] if blocks else []
    m = re.search(r"(\[\s*\{.*\}\s*\])", text, flags=re.S)
    if m:
        cands.append(m.group(1))
    for c in cands:
        try:
            data = json.loads(c.strip())
        except ValueError:
            continue
        if isinstance(data, list):
            return data, bool(blocks)
    return None, bool(blocks)


def check_object(o):
    """Список нарушений схемы для одного объекта (пустой список — объект корректен)."""
    if not isinstance(o, dict):
        return ["не объект"]
    err = []
    if list(sorted(o.keys())) != sorted(KEYS):
        err.append("набор полей")
    nw = lambda s: len(str(s).split())
    if not isinstance(o.get("subject"), str) or not 3 <= nw(o.get("subject", "")) <= 12:
        err.append("subject 3–12 слов")
    if not isinstance(o.get("setting"), str) or not 2 <= nw(o.get("setting", "")) <= 12:
        err.append("setting 2–12 слов")
    if o.get("style") not in STYLES:
        err.append("style вне списка")
    if not isinstance(o.get("lighting"), str) or not 1 <= nw(o.get("lighting", "")) <= 6:
        err.append("lighting 1–6 слов")
    p = o.get("palette")
    if not (isinstance(p, list) and 1 <= len(p) <= 3 and all(x in PALETTE for x in p)):
        err.append("palette: 1–3 цвета из списка")
    if o.get("composition") not in COMPOSITIONS:
        err.append("composition вне списка")
    a = o.get("avoid")
    if not (isinstance(a, list) and all(x in AVOID for x in a)):
        err.append("avoid: элементы из списка")
    if re.search(r"[А-Яа-яЁё]", json.dumps(o, ensure_ascii=False)):
        err.append("кириллица")
    return err


def coverage(o, groups):
    """Доля выполненных групп. Обычная группа — в тексте объекта есть хотя бы одна подстрока; «!цвет» — этого цвета нет в palette;
    «=N» — в palette ровно N цветов."""
    text = json.dumps(o, ensure_ascii=False).lower()
    pal = o.get("palette") if isinstance(o, dict) and isinstance(o.get("palette"), list) else []
    ptxt = " ".join(str(x).lower() for x in pal)
    done = 0
    for g in groups:
        if g[0].startswith("!"):
            done += not any(s[1:] in ptxt for s in g)
        elif g[0].startswith("="):
            done += len(pal) == int(g[0][1:])
        else:
            done += any(s in text for s in g)
    return done / len(groups)


def score_text(text, ctx):
    """K1 — доля объектов из пяти, прошедших проверку схемы (0, если массив не разобран); K2 — среднее покрытие смысловых элементов брифа;
    K3 — доля объектов, у которых список avoid в точности совпал с запретами описания (порядок не важен)."""
    arr, fenced = extract_array(text)
    out = dict(parsed=arr is not None, fenced=fenced, n=0 if arr is None else len(arr), k1=0.0, k2=0.0, k3=0.0, errors=[])
    if arr is None or len(arr) != 5:
        out["errors"].append("массив не разобран" if arr is None else f"объектов {len(arr)} вместо 5")
        if arr is None:
            return out
    ok, cov, exact = 0, [], 0
    for i, o in enumerate(arr[:5]):
        e = check_object(o)
        ok += not e
        out["errors"].extend(f"объект {i + 1}: {x}" for x in e)
        cov.append(coverage(o, BRIEFS[ctx][i][1]))
        exact += isinstance(o, dict) and isinstance(o.get("avoid"), list) and sorted(o["avoid"]) == sorted(AVOID_EXPECTED.get((ctx, i), []))
    cov += [0.0] * (5 - len(cov))
    out["k1"], out["k2"], out["k3"] = ok / 5, sum(cov) / 5, exact / 5
    return out


def score_file(path, ctx):
    return score_text(open(path, encoding="utf-8").read(), ctx)


def stability(paths):
    """K4 — стабильность выбора между повторами одного приёма: доля совпадений полей style и composition по всем парам повторов и всем пяти объектам.
    Учитываются только ответы, из которых разобран массив из пяти объектов."""
    arrs = [a for a, _ in (extract_array(open(p, encoding="utf-8").read()) for p in paths) if a is not None and len(a) == 5]
    same = tot = 0
    for i in range(len(arrs)):
        for j in range(i + 1, len(arrs)):
            for k in range(5):
                for field in ("style", "composition"):
                    same += arrs[i][k].get(field) == arrs[j][k].get(field)
                    tot += 1
    return same / tot if tot else float("nan")


if __name__ == "__main__":
    # python lab09_llm.py prompt zero 1   — напечатать запрос;  python lab09_llm.py score answer.txt 1 — оценить ответ
    if sys.argv[1] == "prompt":
        print(build_prompt(sys.argv[2], int(sys.argv[3])))
    else:
        print(json.dumps(score_file(sys.argv[2], int(sys.argv[3])), ensure_ascii=False, indent=1))

In [ ]:
%%writefile score_all.py
# ЛР 9, вариант 18 (контекст 3): оценка девяти ответов языковой модели. Только стандартная библиотека.
# Файлы: answers/ctx3_<zero|few|step>_r<1..3>.txt (ответ целиком, без правок). Результат: печать таблицы и res9_llm_v18.json
import json, os, sys
import lab09_llm as L

CTX = 3
D = sys.argv[1] if len(sys.argv) > 1 else "answers"
NAMES = {"zero": "zero-shot", "few": "few-shot", "step": "пошаговые инструкции"}


def mean(x):
    return sum(x) / len(x)


def main():
    out, missing = {}, []
    for a in NAMES:
        paths = [os.path.join(D, f"ctx{CTX}_{a}_r{i}.txt") for i in (1, 2, 3)]
        miss = [p for p in paths if not os.path.exists(p)]
        if miss:
            missing += miss
            continue
        rs = [L.score_file(p, CTX) for p in paths]
        out[a] = dict(k1=mean([r["k1"] for r in rs]), k2=mean([r["k2"] for r in rs]), k3=mean([r["k3"] for r in rs]),
                      k4=L.stability(paths), k1_each=[r["k1"] for r in rs], k2_each=[r["k2"] for r in rs],
                      k3_each=[r["k3"] for r in rs], errors=[r["errors"] for r in rs])
    if missing:
        print("нет файлов:", *missing, sep="\n  ")
    print(f"{'приём':22s} {'K1':>5s} {'K2':>6s} {'K3':>5s} {'K4':>6s}")
    for a, r in out.items():
        print(f"{NAMES[a]:22s} {r['k1']:5.2f} {r['k2']:6.3f} {r['k3']:5.2f} {r['k4']:6.3f}")
        for i, e in enumerate(r["errors"], 1):
            if e:
                print(f"   повтор {i}: нарушения:", "; ".join(e))
    json.dump(out, open("res9_llm_v18.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

## Ячейка 3. Сверка библиотек (lab09_lib.py — 7360 байт, SHA-256 cc4d233d…; lab09_llm.py — 18709 байт, 5eabb31f…)

In [ ]:
import hashlib, pathlib
for fn, ref in (("lab09_lib.py", "cc4d233d"), ("lab09_llm.py", "5eabb31f")):
    b = pathlib.Path(fn).read_bytes()
    for name, x in (("как записан", b), ("с \\n в конце", b.rstrip(b"\\n") + b"\\n")):
        h = hashlib.sha256(x).hexdigest(); print(fn, name, len(x), "байт", h[:16] + "…", "— совпадает с эталоном" if h.startswith(ref) else "")

## Ячейка 4. Вариант 18: что сравниваем (шаги 1, 4)

In [ ]:
import json, time, numpy as np, importlib
import lab09_lib as L
importlib.reload(L)
N = 18
c, k = (N - 1) // 6, (N - 1) % 6
PA, PB = L.prompt_a(c), L.prompt_b(c, k)
TARGET = L.ELEMENTS[c][k][1]
OUT = pathlib.Path("artifacts/v18"); OUT.mkdir(parents=True, exist_ok=True)
print("контекст", c + 1, "| элемент", k + 1, L.FACTORS[k])
print("A:", PA); print("B:", PB); print("текст-цель CLIP:", TARGET)
print("B = A + добавленная фраза:", PB == PA + ", " + L.ELEMENTS[c][k][0])
print("режим:", L.MODEL_ID, L.REVISION[:8], "| CLIP", L.CLIP_ID, L.CLIP_REVISION[:8], "| шагов", L.STEPS, "| guidance", L.GUIDANCE, "| размер", L.SIZE, "| seed", list(L.SEEDS))

## Ячейка 5. Загрузка моделей и серия A/B на seed 101…105 (как L.variant_summary, но с сохранением всех десяти изображений)

In [ ]:
t0 = time.perf_counter(); pipe = L.load_pipe(); clip = L.load_clip(); print(f"модели загружены за {time.perf_counter() - t0:.0f} с")
L.generate(pipe, PA, 0)  # прогрев, как в variant_summary
IMG = {"A": {}, "B": {}}; RA, RB = [], []
t0 = time.perf_counter()
for s in L.SEEDS:
    r, img = L.measure(pipe, clip, c, PA, s); RA.append(r); IMG["A"][s] = img; img.save(OUT / f"A_seed{s}.jpg", quality=92)
    r, img = L.measure(pipe, clip, c, PB, s); RB.append(r); IMG["B"][s] = img; img.save(OUT / f"B_seed{s}.jpg", quality=92)
    print("seed", s, "готов, с начала", f"{time.perf_counter() - t0:.0f} с")
ROWS = L.compare(RA, RB, k)
print("среднее время одного изображения, с:", round(float(np.mean([r["seconds"] for r in RA + RB])), 1))

## Ячейка 6. Таблица метрик, признак 2s и допуск (допуски записаны до запуска)

In [ ]:
TOL = {"clip_target": ("abs", 1.0), "clip_subject": ("abs", 1.0), "edge": ("rel", 0.10), "contrast": ("rel", 0.10)}
def classify(m, a, d):
    kind, t = TOL[m]; x = d if kind == "abs" else d / a
    return "в допуске" if abs(x) <= t else ("рост за допуском" if x > 0 else "падение за допуском")
TABLE = []
print(f"{'метрика':13s} {'A':>9s} {'s':>8s} {'B':>9s} {'Δ':>9s} {'|Δ|/s':>7s}  2s   допуск")
for m, a, s, b, d, q, sig in ROWS:
    cl = classify(m, a, d); TABLE.append(dict(metric=m, A=a, s=s, B=b, d=d, ratio=q, sig2s=sig, tolerance=cl, rel=d / a))
    nd = 4 if m in ("edge", "contrast") else 2
    print(f"{m:13s} {a:9.{nd}f} {s:8.{nd}f} {b:9.{nd}f} {d:+9.{nd}f} {q:7.1f}  {'да ' if sig else 'нет'}  {cl}   (отн. {d / a * 100:+.1f} %)")

## Ячейка 7. По каждому seed (диагностика ошибки 1) и знаковая сводка

In [ ]:
pick = {"clip_target": lambda r: r["sims"][1 + k], "clip_subject": lambda r: r["sims"][0], "edge": lambda r: r["edge"], "contrast": lambda r: r["contrast"]}
PER_SEED = {}
for m in pick:
    A = [pick[m](r) for r in RA]; B = [pick[m](r) for r in RB]; PER_SEED[m] = dict(A=A, B=B, diff=[b - a for a, b in zip(A, B)])
    nd = 4 if m in ("edge", "contrast") else 2
    print(m); print("  seed  ", *[f"{s:>9d}" for s in L.SEEDS]); print("  A     ", *[f"{x:9.{nd}f}" for x in A]); print("  B     ", *[f"{x:9.{nd}f}" for x in B])
    print("  B − A ", *[f"{x:+9.{nd}f}" for x in PER_SEED[m]["diff"]], f"| Δ>0 на {sum(x > 0 for x in PER_SEED[m]['diff'])} из 5 seed")
print("\nвсе семь сходств CLIP (предмет + шесть целей) на seed 101, A и B:")
print("  A:", [round(x, 2) for x in RA[0]["sims"]]); print("  B:", [round(x, 2) for x in RB[0]["sims"]])

## Ячейка 8. Листы сравнения: comparison.png (seed 101) и сетка всех десяти изображений

In [ ]:
from PIL import Image, ImageDraw
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10.4, 5.6))
for a_, key, ttl in ((ax[0], "A", "A: " + PA + " (seed 101)"), (ax[1], "B", "B: …, no logo, no text (seed 101)")):
    a_.imshow(IMG[key][101]); a_.set_title(ttl, fontsize=10); a_.axis("off")
plt.tight_layout(); plt.savefig(OUT / "comparison_variant18.png", dpi=110); plt.close()
fig, ax = plt.subplots(2, 5, figsize=(17, 7.4))
for j, s in enumerate(L.SEEDS):
    for i, key in enumerate("AB"):
        ax[i, j].imshow(IMG[key][s]); ax[i, j].set_title(f"{key}, seed {s}", fontsize=10); ax[i, j].axis("off")
plt.tight_layout(); plt.savefig(OUT / "grid_all_seeds.png", dpi=100); plt.close()
from IPython.display import display
display(Image.open(OUT / "comparison_variant18.png")); display(Image.open(OUT / "grid_all_seeds.png"))
json.dump(dict(variant=N, prompt_a=PA, prompt_b=PB, target=TARGET, table=TABLE, per_seed=PER_SEED, seconds=[r["seconds"] for r in RA + RB],
               versions=dict(torch=torch.__version__, diffusers=diffusers.__version__, transformers=transformers.__version__)),
          open(OUT / "summary_v18.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## Ячейка 9. Часть 2: запросы для языковой модели
Девять запросов (3 приёма × 3 повтора) напечатаны ниже и лежат в `prompts/`. **Каждый — в НОВЫЙ чат**, поиск отключён самим запросом.
Ответ сохраняется ЦЕЛИКОМ и без правок в файл `ctx3_<zero|few|step>_r<1..3>.txt`.

In [ ]:
for a in ("zero", "few", "step"):
    print("=" * 30, a, "=" * 30); print(__import__("lab09_llm").build_prompt(a, 3)); print()

## Ячейка 10. Загрузка девяти файлов ответов и оценка K1–K4 (после выполнения части 2)

In [ ]:
from google.colab import files
os.makedirs("answers", exist_ok=True)
up = files.upload()   # выберите девять файлов ctx3_*_r*.txt
for name, data in up.items(): open(os.path.join("answers", name), "wb").write(data)
!python score_all.py answers

## Ячейка 11. Упаковка артефактов

In [ ]:
import shutil, glob
from google.colab import files
for f in glob.glob("res9_llm_v18.json"): shutil.copy(f, OUT)
if os.path.isdir("answers"): shutil.copytree("answers", OUT / "answers", dirs_exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M")
shutil.make_archive(f"lab09_v18_artifacts_{stamp}", "zip", OUT)
files.download(f"lab09_v18_artifacts_{stamp}.zip")